In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
from rich import print

from dotenv import load_dotenv

load_dotenv(override=True)

#0. LLM モデルに接続
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
class OverAllState(MessagesState):
    username: str
    output: str


#2. ノードを定義
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("こんにちは、私は" + state["username"])],
    }


def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#4. グラフを実行
result = graph.invoke({"username": "田中"})
print(result)
